# Imports and Load Prepared Data

In [ ]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
import joblib
import json
import time

X_train_res   = np.load('../models/X_train_res.npy')
y_train_res   = np.load('../models/y_train_res.npy')
X_test_scaled = np.load('../models/X_test_scaled.npy')
y_test        = np.load('../models/y_test.npy')

with open('../models/feature_columns.json', 'r') as f:
    feature_cols = json.load(f)

print(f"All data loaded successfully!")
print(f"  Training samples: {len(X_train_res):,}")
print(f"  Test samples:     {len(X_test_scaled):,}")
print(f"  Features:         {len(feature_cols)}")
print(f"\nTraining class balance:")
print(f"  Stayed  (0): {(y_train_res == 0).sum():,}")
print(f"  Churned (1): {(y_train_res == 1).sum():,}")

All data loaded successfully!
  Training samples: 8,260
  Test samples:     1,407
  Features:         23

Training class balance:
  Stayed  (0): 4,130
  Churned (1): 4,130


# Train Logistic Regression

In [2]:
# Train the model
# WHY LOGISTIC REGRESSION FIRST?
# ✓ Fast to train (seconds, not minutes)
# ✓ Interpretable — you can explain each prediction to a business stakeholder
# ✓ Performs surprisingly well on structured tabular data
# ✓ Standard industry starting point for binary classification
#
# ANALOGY: A weighted scorecard — each feature gets a weight (coefficient)
# Add up all weighted scores → convert to a probability via sigmoid curve
# If probability > 0.5 → predict churn

model = LogisticRegression(max_iter=1000, random_state=42)

print(f"Training model...")
start = time.time()
model.fit(X_train_res, y_train_res)
elapsed = time.time() - start

print(f"Training complete!")
print(f"  Time taken:       {elapsed:.2f} seconds")
print(f"  Model type:       {type(model).__name__}")
print(f"  Training samples: {len(X_train_res):,}")
print(f"  Features used:    {model.n_features_in_}")
print(f"  Iterations run:   {model.n_iter_[0]}")

joblib.dump(model, '../models/churn_model.pkl')
print(f"\nModel saved → models/churn_model.pkl")

Training model...
Training complete!
  Time taken:       0.02 seconds
  Model type:       LogisticRegression
  Training samples: 8,260
  Features used:    23
  Iterations run:   30

Model saved → models/churn_model.pkl


# Make Predictions

In [3]:
# Predict on the test set (data the model has NEVER seen before)

# predict() → hard 0 or 1 decision for each customer
y_pred  = model.predict(X_test_scaled)

# predict_proba() → probability between 0.0 and 1.0
# [:,1] = column 1 = probability of churning (class 1)
# More useful than hard 0/1 — lets us set custom thresholds in the dashboard
y_proba = model.predict_proba(X_test_scaled)[:, 1]

total_churn  = y_pred.sum()
total_stayed = len(y_pred) - total_churn

print(f"Predictions on {len(y_pred):,} test customers:")
print(f"  Predicted to stay:  {total_stayed:,} ({total_stayed/len(y_pred)*100:.1f}%)")
print(f"  Predicted to churn: {total_churn:,} ({total_churn/len(y_pred)*100:.1f}%)")
print(f"\nSample predictions (first 10 customers):")
print(f"  {'#':<5} {'Probability':>12} {'Predicted':>12} {'Actual':>8} {'Match'}")
print(f"  {'-'*48}")
for i in range(10):
    pred_label   = 'CHURN' if y_pred[i] == 1 else 'stay'
    actual_label = 'CHURN' if y_test[i] == 1 else 'stay'
    match        = 'correct' if y_pred[i] == y_test[i] else 'WRONG'
    print(f"  {i+1:<5} {y_proba[i]:>12.3f} {pred_label:>12} {actual_label:>8}  {match}")

Predictions on 1,407 test customers:
  Predicted to stay:  822 (58.4%)
  Predicted to churn: 585 (41.6%)

Sample predictions (first 10 customers):
  #      Probability    Predicted   Actual Match
  ------------------------------------------------
  1            0.035         stay     stay  correct
  2            0.805        CHURN     stay  WRONG
  3            0.009         stay     stay  correct
  4            0.399         stay    CHURN  WRONG
  5            0.234         stay     stay  correct
  6            0.733        CHURN    CHURN  correct
  7            0.056         stay     stay  correct
  8            0.338         stay     stay  correct
  9            0.861        CHURN    CHURN  correct
  10           0.030         stay     stay  correct
